<a href="https://colab.research.google.com/github/saiKelkar/Property_Splats/blob/main/03_my_room.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ===== SESSION START =====
PROJECT = "my_room"
ITERATIONS = 30000

from google.colab import drive
drive.mount('/content/drive')
import os, shutil, glob, psutil
SPLATS = "/content/drive/MyDrive/splats"
DRIVE_DIR = f"{SPLATS}/{PROJECT}"
WORK = f"/content/{PROJECT}"
os.makedirs(WORK, exist_ok=True)

ram_gb = psutil.virtual_memory().total / 1e9
JOBS = max(1, min(os.cpu_count(), int(ram_gb // 6)))
print(f"RAM: {ram_gb:.0f} GB | CPUs: {os.cpu_count()} | compile jobs: {JOBS}")
!nvidia-smi --query-gpu=name,memory.total --format=csv

!apt-get -qq update > /dev/null && apt-get -qq install -y colmap ffmpeg > /dev/null
!pip -q install nerfstudio

wheels = sorted(glob.glob(f"{SPLATS}/wheels/gsplat-*.whl"))
if wheels:
    !pip -q install --no-deps --force-reinstall "{wheels[-1]}"
    print("Installed pre-built gsplat from Drive")
else:
    print("No gsplat wheel in Drive: gsplat will compile on first use (slow)")

!python -c "from gsplat import csrc; print('gsplat ready (pre-built):', csrc is not None)"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
RAM: 55 GB | CPUs: 8 | compile jobs: 8
name, memory.total [MiB]
Tesla T4, 15360 MiB
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Installed pre-built gsplat from Drive
gsplat ready (pre-built): True


In [2]:
shutil.copy(f"{DRIVE_DIR}/input.mp4", f"{WORK}/input.mp4")
print("Video copied:", os.path.getsize(f"{WORK}/input.mp4") // 1_000_000, "MB")

Video copied: 207 MB


In [3]:
os.environ["QT_QPA_PLATFORM"] = "offscreen"
!nohup ns-process-data video --data {WORK}/input.mp4 --output-dir {WORK}/processed --num-frames-target 300 --no-gpu > "{DRIVE_DIR}/process_log.txt" 2>&1 &
print("Processing started in the background")

Processing started in the background


In [5]:
!tail -n 5 "{DRIVE_DIR}/process_log.txt"

Number of frames in video: 968
Extracting 323 frames in evenly spaced intervals


In [19]:
running = !pgrep -f "ns-process-data|colmap"
print("⏳ Still running" if running else "✅ Finished")
!tail -n 5 "{DRIVE_DIR}/process_log.txt"
!grep -i "found poses" "{DRIVE_DIR}/process_log.txt"

✅ Finished
           We extracted 323 images with prefix 'frame_'                               video_to_nerfstudio_dataset.py:152
           Colmap matched 2 images                                                    video_to_nerfstudio_dataset.py:152
           COLMAP only found poses for 0.62% of the images. This is low.              video_to_nerfstudio_dataset.py:152
           This can be caused by a variety of reasons, such poor scene coverage,                                        
           blurry images, or large exposure changes.                                                                    
           COLMAP only found poses for 0.62% of the images. This is low.              video_to_nerfstudio_dataset.py:152


In [20]:
!ls {WORK}/processed/colmap/sparse
!for d in {WORK}/processed/colmap/sparse/*/; do echo "== $d"; colmap model_analyzer --path $d 2>&1 | grep -E "Registered images|Points:|reprojection"; done

0
== {WORK}/processed/colmap/sparse/*/


In [21]:
import subprocess, sqlite3
model = f"{WORK}/processed/colmap/sparse/0"
out = subprocess.run(["colmap", "model_analyzer", "--path", model], capture_output=True, text=True)
print("\n".join(l for l in (out.stdout + out.stderr).splitlines() if any(k in l for k in ["Registered", "Points", "reprojection"])))

db = sqlite3.connect(f"{WORK}/processed/colmap/database.db")
print("Images in database:     ", db.execute("SELECT COUNT(*) FROM images").fetchone()[0])
print("Avg keypoints per image:", round(db.execute("SELECT AVG(rows) FROM keypoints").fetchone()[0]))
print("Image pairs that matched:", db.execute("SELECT COUNT(*) FROM two_view_geometries WHERE rows >= 15").fetchone()[0])

I20261123 06:34:09.196692 58891 model.cc:431] Registered images: 2
I20261123 06:34:09.196700 58891 model.cc:433] Points: 216
I20261123 06:34:09.196735 58891 model.cc:441] Mean reprojection error: 0.660000px
Images in database:      323
Avg keypoints per image: 9854
Image pairs that matched: 1911


In [22]:
names = {1: "degenerate", 2: "calibrated (good 3D)", 3: "uncalibrated (good 3D)", 4: "planar (flat wall)",
         5: "panoramic (rotation only)", 6: "planar or panoramic", 7: "watermark", 8: "multiple"}
for cfg, n in db.execute("SELECT config, COUNT(*) FROM two_view_geometries WHERE rows >= 15 GROUP BY config"):
    print(f"{names.get(cfg, cfg):28s} {n}")

uncalibrated (good 3D)       1690
planar or panoramic          211
watermark                    10


In [23]:
!mkdir -p {WORK}/processed/colmap/sparse_relaxed
!nohup colmap mapper --database_path {WORK}/processed/colmap/database.db --image_path {WORK}/processed/images --output_path {WORK}/processed/colmap/sparse_relaxed --Mapper.init_min_tri_angle 4 --Mapper.init_min_num_inliers 50 --Mapper.abs_pose_min_num_inliers 15 --Mapper.multiple_models 1 > "{DRIVE_DIR}/mapper_relaxed_log.txt" 2>&1 &
print("Relaxed mapper started")

Relaxed mapper started


In [24]:
for m in sorted(glob.glob(f"{WORK}/processed/colmap/sparse_relaxed/*/")):
    out = subprocess.run(["colmap", "model_analyzer", "--path", m], capture_output=True, text=True)
    print("==", m)
    print("\n".join(l for l in (out.stdout + out.stderr).splitlines() if any(k in l for k in ["Registered", "Points:"])))

== /content/my_room/processed/colmap/sparse_relaxed/0/
I20261123 06:34:22.668241 58964 model.cc:431] Registered images: 120
I20261123 06:34:22.668246 58964 model.cc:433] Points: 19062
== /content/my_room/processed/colmap/sparse_relaxed/1/
I20261123 06:34:22.744422 58972 model.cc:431] Registered images: 15
I20261123 06:34:22.744431 58972 model.cc:433] Points: 474


In [35]:
running = !pgrep -f "[c]olmap mapper"
print("⏳ Mapper running" if running else "✅ Mapper finished")
!grep -c "Registering image" "{DRIVE_DIR}/mapper_relaxed_log.txt"
!tail -n 4 "{DRIVE_DIR}/mapper_relaxed_log.txt"

✅ Mapper finished
692
Finding good initial image pair
I20261123 06:41:21.567250 58945 incremental_mapper.cc:404] => No good initial image pair found.
I20261123 06:41:21.568091 58945 timer.cc:91] Elapsed time: 7.046 [minutes]


In [36]:
from nerfstudio.data.utils.colmap_parsing_utils import read_images_binary
for k in ["0", "1"]:
    imgs = read_images_binary(f"{WORK}/processed/colmap/sparse_relaxed/{k}/images.bin")
    nums = sorted(int(i.name.split("_")[1].split(".")[0]) for i in imgs.values())
    print(f"Model {k}: {len(nums)} frames, from frame {nums[0]} to {nums[-1]}")

Model 0: 119 frames, from frame 97 to 259
Model 1: 15 frames, from frame 287 to 301


In [37]:
shutil.copytree(f"{WORK}/processed", f"{DRIVE_DIR}/processed", dirs_exist_ok=True)
print("Processed data saved to Drive")

Processed data saved to Drive


In [38]:
!nohup env MAX_JOBS={JOBS} ns-train splatfacto --output-dir "{DRIVE_DIR}/outputs" --vis tensorboard --max-num-iterations {ITERATIONS} colmap --data {WORK}/processed --colmap-path colmap/sparse_relaxed/0 --downscale-factor 2 > "{DRIVE_DIR}/train_log.txt" 2>&1 &
print("Training started in the background")

Training started in the background


In [69]:
running = !pgrep -f "[n]s-train"
print("⏳ Training" if running else "✅ Finished (or stopped)")
!tail -n 3 "{DRIVE_DIR}/train_log.txt"

✅ Finished (or stopped)
Trainer.train_iteration: 0.0732              
VanillaPipeline.get_train_loss_dict: 0.0677              
Trainer.eval_iteration: 0.0027              


In [70]:
!grep -E "^[0-9]+ \(" "{DRIVE_DIR}/train_log.txt" | tail -n 2
!grep -n -B 25 "AcceleratorError" "{DRIVE_DIR}/train_log.txt" | grep -E "File|Error" | tail -n 15

29990 (99.97%)      102.418 ms           1 s, 24.177 ms       21.31 M               
29999 (100.00%)                                                                     


In [71]:
!grep -m1 "dataparser=" "{DRIVE_DIR}/train_log.txt"
!tail -n 3 "{DRIVE_DIR}/train_log.txt"

            dataparser=ColmapDataParserConfig(
Trainer.train_iteration: 0.0732              
VanillaPipeline.get_train_loss_dict: 0.0677              
Trainer.eval_iteration: 0.0027              


In [72]:
configs = [c for c in glob.glob(f"{DRIVE_DIR}/outputs/**/config.yml", recursive=True)
           if glob.glob(os.path.join(os.path.dirname(c), "nerfstudio_models", "*.ckpt"))]
config = sorted(configs)[-1]
print("Using:", config)
!TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD=1 ns-export gaussian-splat --load-config {config} --output-dir "{DRIVE_DIR}/export"
!ls -lh "{DRIVE_DIR}/export"

Using: /content/drive/MyDrive/splats/my_room/outputs/unnamed/splatfacto/2026-11-23_064220/config.yml
/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
I0000 00:00:1795419648.585578   77857 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/usr/local/lib/python3.13/dist-packages/nerfstudio/field_components/activations.py:32: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @custom_fwd(cast_inputs=torch.float32)
/usr/local/lib/python3.13/dist-packages/nerfstudio/field_components/activations.py:38: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is de